In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))

from dotenv import load_dotenv
from openagv import Project
from openagv.llm import LLMConfig
from openagv.storage import LocalStorageBackend

# Load environment variables
load_dotenv()

# Create project with LLM config for the executor (tool-calling model)
project = Project(
    name="Flowers with Overlays",
    llm_config=LLMConfig(provider="ollama", model="qwen2.5:14b"),
    storage=LocalStorageBackend(root="./project_data", project_id="sample3"),
    width=1920,
    height=1080,
    fps=30.0,
)

# Optional: set a custom system prompt (otherwise uses the built-in default)
# project.system_prompt = "You are a video editor specializing in nature documentaries."

print(f"Project '{project.name}' created (id: {project.id[:8]}...)")

Project 'Flowers with Overlays' created (id: 894cf377...)


In [2]:
from openagv.llm import create_clients
from openagv.modules.vision import ORVisionAnalyzer
from openagv.modules.textcard import TextCardGenerator

# Vision model needs its own client (separate from the executor model)
vision_client, _, _ = create_clients("ollama", "llama3.2-vision")

# Register modules on the project
project.register_module(ORVisionAnalyzer(client=vision_client, model="llama3.2-vision"))
project.register_module(TextCardGenerator(output_dir="./text_cards", width=1920, height=1080, font_size=36))

# Add flower images — metadata (dimensions, format) is extracted automatically on ingest
assets = project.add_assets("../assets/*.jpg")
print(f"Added {len(assets)} assets")
for a in assets:
    print(f"  {a.storage_key} — metadata: {a.metadata}")

# Submit the job
job = project.submit(
    """Create a video showcasing flowers with descriptive text overlays.
    
    For each flower image:
    1. First analyze the image to get a description
    2. Generate a lower-third text overlay with the flower name and a short description
    3. Add the flower to the main timeline track (3 seconds each)
    4. Add the text overlay at the same time position on an overlay track
    
    Make sure every flower has both a background clip and a text overlay.
    Sort flowers by how vibrant/colorful they appear.""",
    author="user:notebook",
)

print(f"Job {job.id[:8]}... submitted (status: {job.status.name})")

# run_job actually executes; wait() only blocks until the done event fires
await project.run_job(job)
print(f"Job finished (status: {job.status.name})")

Added 6 assets
  assets/b8ef5dd5efbf3c851db53edc34f38eac7f17d3000dfe478ded645d7174498796.jpg — metadata: {'width': 2848, 'height': 4288, 'format': 'JPEG'}
  assets/6786e06d6f727b5b8cf39ecb41c9f61ca4fe2763ff8ea066d7ce25c3f8845690.jpg — metadata: {'width': 2092, 'height': 2801, 'format': 'JPEG'}
  assets/25301e980a702dcb97c2376dae14f9e73b56fee9fb969a464aa4ecf277ce8a31.jpg — metadata: {'width': 7952, 'height': 5304, 'format': 'JPEG'}
  assets/61415dce13017e2f4dcea0043b1ba13bc2929d151c13ad2b61937501d5718f20.jpg — metadata: {'width': 3264, 'height': 4896, 'format': 'JPEG'}
  assets/bbd4872365cab6c1e5c099270e047a514cc3f15d6902ccdaac88bb01e444b958.jpg — metadata: {'width': 2333, 'height': 3490, 'format': 'JPEG'}
  assets/b0ea3882b9aba6470a20dd2f05b77caa18b354226020e9e1ed9d2bb20f4ca806.jpg — metadata: {'width': 5261, 'height': 3507, 'format': 'JPEG'}
Job b2ea0f18... submitted (status: PENDING)
Job finished (status: COMPLETED)


In [ ]:
# Inspect timeline state via project
print(project.timeline.get_summary())
print(f"\nOverlay tracks: {project.timeline.get_overlay_tracks()}")

# Per-job chat history (this job's messages only)
print(f"\nJob chat history ({len(job.chat_history)} messages):")
for msg in job.chat_history:
    preview = f"{msg.content[:80]}..." if len(msg.content) > 80 else msg.content
    print(f"  [{msg.role}] ({msg.author}): {preview}")

# Project-level persistent chat history (accumulates across jobs)
print(f"\nProject chat history ({len(project.chat_history)} messages):")
for msg in project.chat_history:
    preview = f"{msg.content[:80]}..." if len(msg.content) > 80 else msg.content
    print(f"  [{msg.role}] ({msg.author}): {preview}")

In [4]:
# Submit a follow-up job on the same project.
# The executor will be seeded with project.chat_history so the LLM
# has context from the previous job.
followup = project.submit(
    "Please verify that every flower in the timeline has a corresponding text overlay. "
    "If any are missing, add them now.",
    author="user:notebook",
)
await project.run_job(followup)
print(f"Follow-up finished (status: {followup.status.name})")
print(project.timeline.get_summary())

# Project chat history now includes both jobs
print(f"\nProject chat history: {len(project.chat_history)} messages across {len(project.jobs)} jobs")

[DEBUG] ORVisionAnalyzer analyzing ./project_data/sample3/assets/b8ef5dd5efbf3c851db53edc34f38eac7f17d3000dfe478ded645d7174498796.jpg with llama3.2-vision...
[DEBUG] ORVisionAnalyzer analyzing ./project_data/sample3/assets/6786e06d6f727b5b8cf39ecb41c9f61ca4fe2763ff8ea066d7ce25c3f8845690.jpg with llama3.2-vision...
[DEBUG] ORVisionAnalyzer analyzing ./project_data/sample3/assets/25301e980a702dcb97c2376dae14f9e73b56fee9fb969a464aa4ecf277ce8a31.jpg with llama3.2-vision...
[DEBUG] ORVisionAnalyzer analyzing ./project_data/sample3/assets/61415dce13017e2f4dcea0043b1ba13bc2929d151c13ad2b61937501d5718f20.jpg with llama3.2-vision...
[DEBUG] ORVisionAnalyzer analyzing ./project_data/sample3/assets/bbd4872365cab6c1e5c099270e047a514cc3f15d6902ccdaac88bb01e444b958.jpg with llama3.2-vision...
[DEBUG] ORVisionAnalyzer analyzing ./project_data/sample3/assets/b0ea3882b9aba6470a20dd2f05b77caa18b354226020e9e1ed9d2bb20f4ca806.jpg with llama3.2-vision...
Follow-up finished (status: RUNNING)
Timeline 'Main 

In [ ]:
# Export timeline as OTIO file
project.timeline.to_otio_file("flowers_overlay.otio")
print("Timeline saved to flowers_overlay.otio")

In [ ]:
# Render using the project's storage backend
from openagv.renderer import FfmpegOTIORenderer

renderer = FfmpegOTIORenderer(storage=project.storage)
renderer.set_otio(project.timeline)
renderer.validate()
renderer.render("flowers_with_overlays.mp4", store_key="renders/flowers.mp4")

In [ ]:
# Serialize the entire project to JSON (for storage in JSONB, etc.)
import json

project_data = project.to_dict()
print(f"Project serialized: {len(json.dumps(project_data))} chars")
print(f"  Assets: {len(project_data['asset_bin']['assets'])}")
print(f"  Jobs: {len(project_data['jobs'])}")
print(f"  Modules: {project_data['modules']}")
print(f"  System prompt: {'custom' if project_data['system_prompt'] else 'default'}")
print(f"  Chat history: {len(project_data['chat_history'])} messages")

# Round-trip: deserialize back (api_key re-injected at load time, storage re-provided)
restored = Project.from_dict(project_data, storage=project.storage)
print(f"\nRestored project '{restored.name}' with {len(restored.asset_bin.assets)} assets")
print(f"  Chat history survived: {len(restored.chat_history)} messages")
print(f"  System prompt survived: {restored.system_prompt!r}")
print(f"  Timeline: {restored.timeline.get_summary()}")